In [ ]:
# notebook cells, roughly:
# 1. load df_rg, region_by_id, df_esm
# 2. X_static, y, groups, factory = assemble_everything(...)   # from assemble_features
# 3. full = run_nested_cv(...)                                  # Goal 1: headline AUC
# 4. results = run_group_configs(..., logo + isolation configs) # Goal 2
#    plot_group_rocs(results, y)
# 5. permutation / SHAP                                          # importance
# 6. fit_final_model(...); apply_to_new_set(...)                # Goal 3

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd, numpy as np
import json
from src.classifier.assemble_features import assemble_everything
from src.classifier.feature_groups import FEATURE_GROUPS, SUPERGROUPS
from src.classifier.nested_cv import run_nested_cv, report_ungrouped

RATES_PATH = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/samocha_mutation_rates/fordist_1KG_mutation_rate_table.txt"

# df_rg, region_by_id, df_esm already loaded from your pipeline

In [ ]:
DATA_DIR = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"

df_for_rg = pd.read_parquet(f"{DATA_DIR}/variants_annotated_final.parquet")
print(f"Loaded {len(df_for_rg):,} variant-region assignments")
print(f"Columns ({len(df_for_rg.columns)}): {df_for_rg.columns.tolist()}")


# Also load the regions JSON — useful for context (WT sequences, etc.)
with open(f"{DATA_DIR}/genomic_coords_merged_win5.json") as f:
    regions = json.load(f)
region_by_id = {r["region_id"]: r for r in regions}
print(f"Loaded {len(regions)} regions from JSON")


df_esm = pd.read_parquet(f"{DATA_DIR}/variants_with_esm.parquet")

In [ ]:
X_static, y, groups, factory = assemble_everything(
    df_for_rg, region_by_id, df_esm=df_esm,
    rates_path=RATES_PATH,
    codon_source_aas=list("ADEGLPRS"),
    codon_rates=None,
    alpha=0.5, split_source=False,
    physchem_delta_cache="/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/physchem_deltas.parquet",
    sub_from_col="before_aa",      # <-- match compute_substitution_counts
    sub_to_col="after_aa",         # <-- match compute_substitution_counts
)

print("X_static:", X_static.shape, "| regions:", len(X_static), "| accessions:", groups.nunique())
print("label balance:", y.value_counts().to_dict())
print("\nungrouped columns (in matrix, in NO group — would never be tested):")
print(report_ungrouped(X_static.columns))

In [ ]:
full = run_nested_cv(
    X_static, y, groups,
    include_groups=list(FEATURE_GROUPS),
    folded_transformer_factory=factory,
    n_splits=5, n_trees=300,
)
print(f"FULL model: AUC {full['mean_auc']:.3f} ± {full['std_auc']:.3f}")
print("per-fold:", [round(a,3) for a in full['per_fold_auc']])
print("static cols used:", full['n_static_cols'], "| folded used:", full['used_folded'])

In [ ]:
from src.classifier.group_analysis import run_group_comparison, plot_group_rocs, plot_necessity_sufficiency

results, table = run_group_comparison(
    X_static, y, groups, factory=factory,
    n_splits=5, n_trees=300,        # full group set (omit group_names)
)
table.to_csv("/mnt/d/phd/scripts/16_ev_signature_predictor/figures/group_comparison.csv", index=False)

# ROC: full + each group alone (or pick a subset to keep it readable)
iso_labels = ["FULL (all groups)"] + [f"{g} only" for g in FEATURE_GROUPS]
fig = plot_group_rocs(results, y, configs_to_show=iso_labels)
fig.savefig("/mnt/d/phd/scripts/16_ev_signature_predictor/figures/group_roc.png", dpi=200, bbox_inches="tight")

fig2 = plot_necessity_sufficiency(table)
fig2.savefig("/mnt/d/phd/scripts/16_ev_signature_predictor/figures/group_necessity_sufficiency.png", dpi=200, bbox_inches="tight")

In [ ]:
import pickle
path_prefix = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/output/group_RF_comparison"

table.to_parquet(f"{path_prefix}_table.parquet", index=False)
with open(f"{path_prefix}_results.pkl", "wb") as fh:
    pickle.dump(results, fh)
print(f"saved {path_prefix}_table.parquet and {path_prefix}_results.pkl")

In [ ]:
path_prefix = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/output/group_RF_comparison"

table = pd.read_parquet(f"{path_prefix}_table.parquet")
with open(f"{path_prefix}_results.pkl", "rb") as fh:
    results = pickle.load(fh)

In [ ]:
from src.classifier.importance import (
    group_permutation_importance, fit_full_model_for_shap, shap_importance,
    plot_group_importance, plot_shap_summary, plot_shap_bar)

# --- group permutation importance (the headline) ---
gpi = group_permutation_importance(
    X_static, y, groups,
    include_groups=[g for g in FEATURE_GROUPS if g != "substitution_score"],  # fold-static
    n_splits=5, n_trees=300, n_repeats=10)
print(gpi.round(4).to_string(index=False))
fig = plot_group_importance(gpi)
fig.savefig("/mnt/d/phd/scripts/16_ev_signature_predictor/figures/group_permutation_importance.png", dpi=200, bbox_inches="tight")

# --- SHAP (per-feature, for the narrative) ---
rf, Xi_df, names, imp = fit_full_model_for_shap(
    X_static, y,
    include_groups=[g for g in FEATURE_GROUPS if g != "substitution_score"])
sv, shap_summary = shap_importance(rf, Xi_df)
print(shap_summary.head(20).to_string(index=False))
fig = plot_shap_summary(sv, Xi_df)            # beeswarm
fig.savefig("/mnt/d/phd/scripts/16_ev_signature_predictor/figures/shap_beeswarm.png", dpi=200, bbox_inches="tight")

In [ ]:
features_df

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Feature selection pipeline: correlation pruning + RFECV with grouped CV
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# ── Config ──────────────────────────────────────────────────────────────
RANDOM_STATE = 42
N_TREES = 100
CORR_THRESHOLD = 0.95
N_SPLITS = 5

POS_COLOR = "#4daf4a"
NEG_COLOR = "#e41a1c"

GROUP_COL = "protein_id"   # ← change to your actual protein/cluster column

# ── Prep ────────────────────────────────────────────────────────────────
all_features = [c for c in features_df.columns if c not in EXCLUDE_COLS]
numeric_features = features_df[all_features].select_dtypes(include=[np.number]).columns.tolist()
dropped_nonnumeric = set(all_features) - set(numeric_features)
if dropped_nonnumeric:
    print(f"Dropped {len(dropped_nonnumeric)} non-numeric features: {sorted(dropped_nonnumeric)}")

X_df = features_df[numeric_features].copy()
y = (features_df["group"] == "pos").astype(int).values
groups = features_df[GROUP_COL].values

print(f"Starting with {X_df.shape[1]} numeric features, "
      f"{X_df.shape[0]} regions, {len(np.unique(groups))} unique groups")

# ── Step 1: Drop near-zero-variance features ───────────────────────────
variances = X_df.var()
near_zero = variances[variances < 1e-8].index.tolist()
if near_zero:
    print(f"\n[Step 1] Dropping {len(near_zero)} near-zero-variance features: {near_zero}")
    X_df = X_df.drop(columns=near_zero)

# ── Step 2: Correlation pruning ─────────────────────────────────────────
def prune_correlated(df, threshold=0.95):
    """Drop one of each highly correlated pair; keep the higher-variance one."""
    corr = df.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = set()
    for col in upper.columns:
        if col in to_drop:
            continue
        correlated = upper.index[upper[col] > threshold].tolist()
        for partner in correlated:
            if partner in to_drop:
                continue
            # Keep whichever has more variance
            loser = partner if df[col].var() >= df[partner].var() else col
            to_drop.add(loser)
    return [c for c in df.columns if c not in to_drop], sorted(to_drop)

kept, pruned = prune_correlated(X_df, threshold=CORR_THRESHOLD)
print(f"\n[Step 2] Correlation pruning at |r| > {CORR_THRESHOLD}:")
print(f"  Kept: {len(kept)}    Dropped: {len(pruned)}")
if pruned:
    print(f"  Dropped features: {pruned}")
X_df = X_df[kept]

# ── Step 3: Baseline CV AUC on pruned set ──────────────────────────────
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
baseline_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1)),
])
baseline_auc = cross_val_score(baseline_pipe, X_df.values, y, groups=groups,
                                cv=cv, scoring="roc_auc", n_jobs=-1)
print(f"\n[Step 3] Baseline CV AUC after pruning: "
      f"{baseline_auc.mean():.3f} ± {baseline_auc.std():.3f}")

# ── Step 4: RFECV with grouped CV ──────────────────────────────────────
print(f"\n[Step 4] Running RFECV (this may take a few minutes)...")

# Impute once for RFECV (it doesn't accept a pipeline as estimator)
imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X_df.values)

rfecv = RFECV(
    estimator=RandomForestClassifier(n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1),
    step=5,
    min_features_to_select=5,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
)
rfecv.fit(X_imputed, y, groups=groups)

selected_features = [f for f, keep in zip(X_df.columns, rfecv.support_) if keep]
print(f"  RFECV selected {rfecv.n_features_} features (out of {X_df.shape[1]})")
print(f"  CV AUC at optimum: {rfecv.cv_results_['mean_test_score'][rfecv.n_features_ // 5]:.3f}")

# ── Step 5: Visualize ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: AUC vs number of features
ax = axes[0]
n_features_grid = np.arange(1, len(rfecv.cv_results_["mean_test_score"]) + 1) * 5
n_features_grid = n_features_grid[:len(rfecv.cv_results_["mean_test_score"])]
mean_scores = rfecv.cv_results_["mean_test_score"]
std_scores = rfecv.cv_results_["std_test_score"]

ax.plot(n_features_grid, mean_scores, "-o", color=POS_COLOR, markersize=4, linewidth=1.2)
ax.fill_between(n_features_grid, mean_scores - std_scores, mean_scores + std_scores,
                color=POS_COLOR, alpha=0.2)
ax.axvline(rfecv.n_features_, color=NEG_COLOR, linestyle="--", linewidth=1,
           label=f"Optimum: {rfecv.n_features_} features")
ax.set_xlabel("Number of features")
ax.set_ylabel("CV AUC (mean ± std)")
ax.set_title("RFECV: AUC vs feature count")
ax.legend(frameon=False)
ax.grid(alpha=0.3, linestyle=":", linewidth=0.4)

# Right: which AM features survived
ax = axes[1]
am_in_selected = [f for f in selected_features if f in AM_FEATURES]
am_dropped = [f for f in AM_FEATURES if f not in selected_features and f in X_df.columns]
categories = ["AM kept", "AM dropped", "Non-AM kept"]
counts = [len(am_in_selected), len(am_dropped), len(selected_features) - len(am_in_selected)]
colors = ["tab:orange", "lightgray", "tab:blue"]
ax.bar(categories, counts, color=colors, edgecolor="black", linewidth=0.4)
for i, c in enumerate(counts):
    ax.text(i, c + 0.3, str(c), ha="center", fontsize=10)
ax.set_ylabel("Count")
ax.set_title("Feature survival by group")
ax.grid(axis="y", alpha=0.3, linestyle=":", linewidth=0.4)

for ax in axes:
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

plt.tight_layout()
plt.show()

# ── Step 6: Final comparison ───────────────────────────────────────────
final_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1)),
])
final_auc = cross_val_score(final_pipe, X_df[selected_features].values, y, groups=groups,
                             cv=cv, scoring="roc_auc", n_jobs=-1)

print("\n" + "═" * 65)
print("SUMMARY")
print("═" * 65)
print(f"{'Stage':<35} {'N features':>12} {'CV AUC':>15}")
print("─" * 65)
print(f"{'Original':<35} {len(numeric_features):>12} {'(see prev run)':>15}")
print(f"{'After correlation pruning':<35} {len(kept):>12} "
      f"{baseline_auc.mean():.3f} ± {baseline_auc.std():.3f}")
print(f"{'After RFECV':<35} {rfecv.n_features_:>12} "
      f"{final_auc.mean():.3f} ± {final_auc.std():.3f}")

print(f"\nSelected features ({len(selected_features)}):")
for f in selected_features:
    marker = " [AM]" if f in AM_FEATURES else ""
    print(f"  • {f}{marker}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Quick RF classifier test: with AM only / without AM / with all features
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score, roc_curve,
)
from sklearn.impute import SimpleImputer

# ── Setup ────────────────────────────────────────────────────────────────
RANDOM_STATE = 42
N_TREES = 100

EXCLUDE_COLS = {"region_id", "group", "label", "group_x", "group_y"}
AM_FEATURES = [c for c in features_df.columns
               if "alphamissense" in c.lower() or c.startswith("am_")
               or c == "fraction_pathogenic"]

print(f"Total features: {len(features_df.columns) - len(EXCLUDE_COLS)}")
print(f"AlphaMissense features ({len(AM_FEATURES)}): {AM_FEATURES}")
print(f"Dataset: {len(features_df)} regions "
      f"({(features_df['group'] == 'pos').sum()} pos, "
      f"{(features_df['group'] == 'neg').sum()} neg)")

y = (features_df["group"] == "pos").astype(int).values

def _run_rf(feature_subset, label):
    # Keep only numeric features
    X_df = features_df[feature_subset].select_dtypes(include=[np.number])
    dropped = set(feature_subset) - set(X_df.columns)
    if dropped:
        print(f"  (Dropped {len(dropped)} non-numeric features: {sorted(dropped)})")
    feature_subset = list(X_df.columns)
    X = X_df.values

    imputer = SimpleImputer(strategy="median")
    X_imputed = imputer.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_imputed, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
    )
    rf = RandomForestClassifier(
        n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1,
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    y_proba = rf.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_auc = cross_val_score(
        RandomForestClassifier(n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1),
        X_imputed, y, scoring="roc_auc", cv=cv,
    )
    cv_acc = cross_val_score(
        RandomForestClassifier(n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1),
        X_imputed, y, scoring="accuracy", cv=cv,
    )

    rf_full = RandomForestClassifier(
        n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1,
    )
    rf_full.fit(X_imputed, y)
    importances = pd.Series(
        rf_full.feature_importances_, index=feature_subset,
    ).sort_values(ascending=False)

    print(f"\n── {label} ────────────────────────────────────")
    print(f"  Features: {len(feature_subset)}")
    print(f"  Test set (80/20): accuracy = {acc:.3f}, AUC = {auc:.3f}")
    print(f"  5-fold CV:        accuracy = {cv_acc.mean():.3f} ± {cv_acc.std():.3f}, "
          f"AUC = {cv_auc.mean():.3f} ± {cv_auc.std():.3f}")
    print(f"\n  Top 15 features by importance:")
    print(importances.head(15).to_string())

    return {
        "label": label,
        "test_accuracy": acc,
        "test_auc": auc,
        "cv_accuracy_mean": cv_acc.mean(),
        "cv_accuracy_std": cv_acc.std(),
        "cv_auc_mean": cv_auc.mean(),
        "cv_auc_std": cv_auc.std(),
        "importances": importances,
        "y_test": y_test,
        "y_proba": y_proba,
    }
# ── Run three configurations ────────────────────────────────────────────
all_features = [c for c in features_df.columns if c not in EXCLUDE_COLS]
features_without_am = [c for c in all_features if c not in AM_FEATURES]
features_only_am = list(AM_FEATURES)

result_all = _run_rf(all_features, "ALL features (with AM)")
result_without_am = _run_rf(features_without_am, "WITHOUT AlphaMissense")
result_only_am = _run_rf(features_only_am, "AM ONLY")


# ── Comparative visualization ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: ROC curves for all three configurations
ax = axes[0]
styles = [("-", "tab:blue"), ("--", "tab:green"), (":", "tab:orange")]
for r, (linestyle, color) in zip(
    [result_all, result_without_am, result_only_am], styles,
):
    fpr, tpr, _ = roc_curve(r["y_test"], r["y_proba"])
    ax.plot(fpr, tpr, linestyle=linestyle, color=color, linewidth=1.5,
            label=f"{r['label']} (AUC={r['test_auc']:.3f})")
ax.plot([0, 1], [0, 1], "k:", linewidth=0.6, alpha=0.5)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC comparison")
ax.legend(frameon=False, fontsize=9, loc="lower right")
ax.grid(alpha=0.3, linestyle=":", linewidth=0.4)

# Right: Top-20 feature importances (WITH AM)
ax = axes[1]
top20 = result_all["importances"].head(20)
colors = ["tab:orange" if f in AM_FEATURES else "tab:blue" for f in top20.index]
ax.barh(range(len(top20)), top20.values[::-1],
        color=colors[::-1], edgecolor="black", linewidth=0.3)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20.index[::-1], fontsize=7)
ax.set_xlabel("Feature importance")
ax.set_title("Top 20 features (ALL)  — orange = AM")
ax.grid(axis="x", alpha=0.3, linestyle=":", linewidth=0.4)

for side in ("top", "right"):
    axes[0].spines[side].set_visible(False)
    axes[1].spines[side].set_visible(False)

plt.tight_layout()
plt.show()

# ── Summary ─────────────────────────────────────────────────────────────
print("\n" + "═" * 60)
print("SUMMARY — 5-fold CV")
print("═" * 60)
print(f"{'Configuration':<30} {'CV AUC':>15} {'CV accuracy':>18}")
print("─" * 65)
for r in [result_all, result_without_am, result_only_am]:
    auc_str = f"{r['cv_auc_mean']:.3f} ± {r['cv_auc_std']:.3f}"
    acc_str = f"{r['cv_accuracy_mean']:.3f} ± {r['cv_accuracy_std']:.3f}"
    print(f"{r['label']:<30} {auc_str:>15} {acc_str:>18}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Leave-one-gene-out CV: honest generalization test
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42
N_TREES = 100
EXCLUDE_COLS = {"region_id", "group", "label", "group_x", "group_y"}
AM_FEATURES = [c for c in features_df.columns
               if "alphamissense" in c.lower() or c.startswith("am_")]

# Extract gene (UniProt accession) from region_id
# region_id format: UniProtAccession_start_end
features_df = features_df.copy()
features_df["gene"] = features_df["region_id"].str.split("_").str[0]

n_genes = features_df["gene"].nunique()
region_per_gene = features_df["gene"].value_counts()
print(f"Number of unique genes: {n_genes}")
print(f"Regions per gene: min={region_per_gene.min()}, "
      f"median={region_per_gene.median():.0f}, "
      f"max={region_per_gene.max()}, "
      f"mean={region_per_gene.mean():.2f}")
print(f"Genes with >1 region: {(region_per_gene > 1).sum()}")

y = (features_df["group"] == "pos").astype(int).values
groups = features_df["gene"].values

def _run_logo(feature_subset, label):
    # Keep only numeric
    X_df = features_df[feature_subset].select_dtypes(include=[np.number])
    feature_subset = list(X_df.columns)
    X = X_df.values

    # Impute
    imputer = SimpleImputer(strategy="median")
    X_imputed = imputer.fit_transform(X)

    # Leave-one-gene-out CV
    logo = LeaveOneGroupOut()

    # Get per-region predictions via out-of-fold prediction
    rf = RandomForestClassifier(
        n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1,
    )
    y_proba = cross_val_predict(
        rf, X_imputed, y, groups=groups, cv=logo, method="predict_proba",
        n_jobs=-1,
    )[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    auc = roc_auc_score(y, y_proba)
    acc = accuracy_score(y, y_pred)

    print(f"\n── {label} ────────────────────────────────────")
    print(f"  Features: {len(feature_subset)}")
    print(f"  LOGO-CV accuracy: {acc:.3f}")
    print(f"  LOGO-CV AUC:      {auc:.3f}")

    return {
        "label": label,
        "auc": auc,
        "accuracy": acc,
        "y_true": y,
        "y_proba": y_proba,
    }


all_features = [c for c in features_df.columns
                if c not in EXCLUDE_COLS and c != "gene"]
features_without_am = [c for c in all_features if c not in AM_FEATURES]
features_only_am = list(AM_FEATURES)

result_all = _run_logo(all_features, "ALL features (with AM)")
result_without_am = _run_logo(features_without_am, "WITHOUT AlphaMissense")
result_only_am = _run_logo(features_only_am, "AM ONLY")


# ── ROC comparison figure ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
styles = [("-", "tab:blue"), ("--", "tab:green"), (":", "tab:orange")]
for r, (ls, color) in zip(
    [result_all, result_without_am, result_only_am], styles,
):
    fpr, tpr, _ = roc_curve(r["y_true"], r["y_proba"])
    ax.plot(fpr, tpr, linestyle=ls, color=color, linewidth=1.5,
            label=f"{r['label']} (AUC={r['auc']:.3f})")
ax.plot([0, 1], [0, 1], "k:", linewidth=0.6, alpha=0.5)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("Leave-one-gene-out ROC")
ax.legend(frameon=False, fontsize=9, loc="lower right")
ax.grid(alpha=0.3, linestyle=":", linewidth=0.4)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()


# ── Final comparison vs earlier stratified 5-fold ──────────────────────
print("\n" + "═" * 60)
print("HONEST SUMMARY: stratified 5-fold vs leave-one-gene-out")
print("═" * 60)
print(f"{'Configuration':<30} {'5-fold AUC':>15} {'LOGO AUC':>15}")
print("─" * 60)
print(f"{'ALL features (with AM)':<30} {'0.838':>15} {result_all['auc']:>15.3f}")
print(f"{'WITHOUT AlphaMissense':<30} {'0.797':>15} {result_without_am['auc']:>15.3f}")
print(f"{'AM ONLY':<30} {'0.786':>15} {result_only_am['auc']:>15.3f}")
print("\nIf LOGO numbers are MUCH lower than 5-fold, the classifier was")
print("largely learning gene identity, not generalizable biology.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Feature importance: permutation vs tree-based (MDI), side by side, 
# colored by category
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer

# ── Category palette ──────────────────────────────────────────────────────
# Consistent across both panels for direct comparison
CATEGORY_COLORS = {
    "AlphaMissense":       "#e41a1c",  # red
    "ESM1b LLR":           "#984ea3",  # purple
    "RG burden/events":    "#4daf4a",  # green
    "Substitution class":  "#377eb8",  # blue
    "Physchem Δ":          "#ff7f00",  # orange
    "Physchem WT":         "#ffcc00",  # yellow-orange
    "Codon usage":         "#a65628",  # brown
    "Variant burden":      "#f781bf",  # pink
    "Other":               "#999999",  # grey
}

def _category(f):
    if f.startswith("am_"):       return "AlphaMissense"
    if f.startswith("esm_"):      return "ESM1b LLR"
    if f.startswith("rg_"):       return "RG burden/events"
    if f.startswith("sub_rate_"): return "Substitution class"
    if f.startswith("delta_"):    return "Physchem Δ"
    if f.startswith("wt_"):       return "Physchem WT"
    if f.startswith("codon_"):    return "Codon usage"
    if (f.startswith("density_") or f.startswith("fraction_") 
        or f.startswith("n_")):
        return "Variant burden"
    return "Other"


# ── Prep features ──────────────────────────────────────────────────────────
X_full_df = features_df[all_features].select_dtypes(include=[np.number])
feature_names = X_full_df.columns.tolist()
imputer = SimpleImputer(strategy="median")
X_full = imputer.fit_transform(X_full_df.values)


# ── Permutation importance (5-fold group, mean across folds) ──────────────
print("Computing permutation importance...")
gkf = GroupKFold(n_splits=5)
fold_perm = []
for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(X_full, y, groups)):
    rf_fold = RandomForestClassifier(
        n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1,
    )
    rf_fold.fit(X_full[train_idx], y[train_idx])
    perm = permutation_importance(
        rf_fold, X_full[test_idx], y[test_idx],
        n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1,
        scoring="roc_auc",
    )
    fold_perm.append(perm.importances_mean)
    print(f"  fold {fold_idx+1}/5 done")

perm_imp_mean = np.mean(fold_perm, axis=0)
perm_imp_std = np.std(fold_perm, axis=0)

perm_df = pd.DataFrame({
    "feature": feature_names,
    "importance_mean": perm_imp_mean,
    "importance_std":  perm_imp_std,
    "category":        [_category(f) for f in feature_names],
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)


# ── Tree (MDI) importance — single model fit on all data ──────────────────
print("\nComputing tree-based importance (MDI)...")
rf_full = RandomForestClassifier(
    n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1,
)
rf_full.fit(X_full, y)
mdi_imp = rf_full.feature_importances_

# For uncertainty, also collect per-tree MDI std across the forest
mdi_per_tree = np.array([
    tree.feature_importances_ for tree in rf_full.estimators_
])
mdi_std = mdi_per_tree.std(axis=0)

mdi_df = pd.DataFrame({
    "feature":         feature_names,
    "importance_mean": mdi_imp,
    "importance_std":  mdi_std,
    "category":        [_category(f) for f in feature_names],
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)


# ── Plot: side-by-side, top 25, colored by category ───────────────────────
TOP_N = 25
top_perm = perm_df.head(TOP_N)
top_mdi  = mdi_df.head(TOP_N)

fig, axes = plt.subplots(1, 2, figsize=(13, 9))

def _plot_panel(ax, df_top, title, xlabel):
    y_pos = np.arange(len(df_top))
    colors = [CATEGORY_COLORS[c] for c in df_top["category"]]
    ax.barh(
        y_pos, df_top["importance_mean"],
        xerr=df_top["importance_std"],
        color=colors, alpha=0.85, edgecolor="black", linewidth=0.5,
        error_kw=dict(ecolor="black", lw=0.6, capsize=2),
    )
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_top["feature"], fontsize=7.5)
    ax.invert_yaxis()
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.axvline(0, color="black", linewidth=0.5)
    ax.grid(alpha=0.3, linestyle=":", linewidth=0.4, axis="x")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

_plot_panel(
    axes[0], top_perm,
    title="Permutation importance (5-fold group)",
    xlabel="Δ AUC when permuted",
)
_plot_panel(
    axes[1], top_mdi,
    title="Tree-based importance (MDI)",
    xlabel="Mean decrease in impurity",
)

# Shared category legend at the bottom
present_categories = sorted(set(
    list(top_perm["category"]) + list(top_mdi["category"])
))
legend_handles = [
    plt.Rectangle((0,0), 1, 1, facecolor=CATEGORY_COLORS[c],
                   edgecolor="black", linewidth=0.5, label=c)
    for c in present_categories
]
fig.legend(
    handles=legend_handles,
    loc="lower center", ncol=min(len(present_categories), 5),
    frameon=False, fontsize=8.5,
    bbox_to_anchor=(0.5, -0.01),
)

fig.suptitle(
    "Feature importance for the classifier (top 25)",
    fontsize=12, y=0.995,
)
plt.tight_layout(rect=[0, 0.04, 1, 0.985])
plt.savefig(
    "/mnt/d/phd/scripts/16_ev_signature_predictor/figures/"
    "feature_importance_comparison.svg",
    dpi=600, bbox_inches="tight",
)
plt.savefig(
    "/mnt/d/phd/scripts/16_ev_signature_predictor/figures/"
    "feature_importance_comparison.png",
    dpi=600, bbox_inches="tight",
)
plt.show()


# ── Category-level summary ────────────────────────────────────────────────
print("\n" + "═" * 60)
print("Category contributions (full feature list, not just top 25)")
print("═" * 60)

cat_summary_perm = (
    perm_df.groupby("category")["importance_mean"]
    .agg(["sum", "mean", "count"])
    .sort_values("sum", ascending=False)
)
cat_summary_mdi = (
    mdi_df.groupby("category")["importance_mean"]
    .agg(["sum", "mean", "count"])
    .sort_values("sum", ascending=False)
)

print("\nPermutation importance — by category:")
print(cat_summary_perm.round(4))
print("\nTree importance (MDI) — by category:")
print(cat_summary_mdi.round(4))


# ── Optional: rank correlation between the two methods ────────────────────
from scipy.stats import spearmanr
both = perm_df[["feature", "importance_mean"]].rename(
    columns={"importance_mean": "perm"}
).merge(
    mdi_df[["feature", "importance_mean"]].rename(
        columns={"importance_mean": "mdi"}
    ),
    on="feature",
)
rho, p = spearmanr(both["perm"], both["mdi"])
print(f"\nSpearman correlation between permutation and MDI rankings: "
      f"ρ = {rho:.3f}, p = {p:.2e}")
print("(High ρ = both methods agree on which features matter)")